In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & FILE DISCOVERY
# ==========================================
print("Searching for processed feature files...")

BASE_DIR = Path(os.getcwd())

# Find all files matching "features_*.csv" in the current folder and all subdirectories
# This automatically grabs normal/features_normal.csv and ataque/dos/features_dos.csv, etc.
feature_files = list(BASE_DIR.rglob("features_*.csv"))

if not feature_files:
    raise ValueError(f"No features_*.csv files found in {BASE_DIR} or its subdirectories.")

print(f"Found {len(feature_files)} feature files. Loading data...")

# ==========================================
# 2. MERGE & CHRONOLOGICAL SORT
# ==========================================
dataframes = []
for file in feature_files:
    print(f"  -> Loading {file.name}...")
    df = pd.read_csv(file)
    dataframes.append(df)

# Stitch everything into one massive master dataset
master_df = pd.concat(dataframes, ignore_index=True)

# ==========================================
# 3. STRATIFIED TEMPORAL TRAIN / TEST SPLIT
# ==========================================
print("Executing Stratified Temporal Split...")

train_chunks = []
test_chunks = []
train_ratio = 0.70

# Group by attack category to ensure proportional representation in Train/Test
for category, group_df in master_df.groupby('attack_cat'):
    
    # Sort each category chronologically just to be safe
    group_df = group_df.sort_values('ltime').reset_index(drop=True)
    
    # Calculate the 70% split index for this specific category
    split_idx = int(len(group_df) * train_ratio)
    
    # Append the chronological chunks
    train_chunks.append(group_df.iloc[:split_idx])
    test_chunks.append(group_df.iloc[split_idx:])

# Combine all training chunks and all testing chunks
train_df = pd.concat(train_chunks, ignore_index=True)
test_df = pd.concat(test_chunks, ignore_index=True)

# Shuffle the final sets. 
# Since the model reads data row-by-row during training, we don't want it 
# to read 50,000 Normal rows, then 10,000 Fuzzer rows in solid blocks.
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Training Set Shape: {train_df.shape}")
print(f"Testing Set Shape:  {test_df.shape}")

# Verify the attack distribution in both sets (They should now be nearly identical)
print("\nAttack Category Distribution (Training):")
print(train_df['attack_cat'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

print("\nAttack Category Distribution (Testing):")
print(test_df['attack_cat'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

# ==========================================
# 4. FEATURE SELECTION (PREVENTING DATA LEAKAGE)
# ==========================================

# Reassign a clean, global ID now that everything is sorted
master_df['id'] = range(1, len(master_df) + 1)

print("Dropping network identities and timestamps to prevent ML data leakage...")

# We drop the IPs, Ports, and Timestamps. 
columns_to_drop = ['srcip', 'sport', 'dstip', 'dsport', 'stime', 'ltime']

# Safely drop them if they exist
master_df.drop(columns=[col for col in columns_to_drop if col in master_df.columns], inplace=True)

# ==========================================
# 5. EXPORT
# ==========================================
OUTPUT_DIR = BASE_DIR.parent.parent.parent / "notebooks" / "Ubuntu-Server-Logs-Training-Test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = OUTPUT_DIR/"ubuntu-server_training-set.csv"
TEST_FILE = OUTPUT_DIR/"ubuntu-server_testing-set.csv"

print("\nExporting final ML datasets...")
train_df.to_csv(TRAIN_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

print(f"Done! Datasets successfully saved to {OUTPUT_DIR}")

Searching for processed feature files...
Found 6 feature files. Loading data...
  -> Loading features_normal.csv...
  -> Loading features_analysis.csv...
  -> Loading features_dos.csv...
  -> Loading features_exploits.csv...
  -> Loading features_fuzzers.csv...
  -> Loading features_reconnaissance.csv...
Executing Stratified Temporal Split...
Training Set Shape: (147613, 51)
Testing Set Shape:  (63267, 51)

Attack Category Distribution (Training):
attack_cat
Normal            47.97%
DoS               18.73%
Exploits          12.95%
Reconnaissance     7.59%
Fuzzers            6.46%
Analysis           6.31%
Name: proportion, dtype: object

Attack Category Distribution (Testing):
attack_cat
Normal            47.97%
DoS               18.73%
Exploits          12.95%
Reconnaissance     7.59%
Fuzzers            6.46%
Analysis           6.31%
Name: proportion, dtype: object
Dropping network identities and timestamps to prevent ML data leakage...

Exporting final ML datasets...
Done! Datasets s